In [34]:
!pip -q install asf_search

In [35]:
import os
import asf_search as asf
from google.colab import userdata

try:
    # Added .strip() to remove any accidental trailing spaces or newlines in the token
    token = userdata.get('EARTHDATA').strip()
except Exception as e:
    raise SystemExit(f'Could not read the secret "EARTHDATA": {e}')
session = asf.ASFSession().auth_with_token(token)
print('logged in')

logged in


In [36]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/nisar_project'
for site in ['langtang', 'suicide_basin']:
    for prod in ['GCOV', 'GUNW', 'GOFF']:
        os.makedirs(f'{ROOT}/crops/{site}/{prod}', exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
import pandas as pd
from shapely.geometry import shape, box as shp_box
pd.set_option('display.width', 200); pd.set_option('display.max_rows', 500)

START = '2026-06-17T00:00:00'
END   = '2026-09-30T23:59:59'
MIN_COVER = 95

SITES = {
    'langtang':      (85.0, 27.8, 85.8, 28.5),
    'suicide_basin': (-134.75, 58.35, -134.25, 58.60),
}
PICK = {
    'langtang':      dict(track=98,  direction='ASCENDING'),
    'suicide_basin': dict(track=144, direction='DESCENDING'),
}
PRODUCTS = ['GCOV', 'GUNW', 'GOFF']

def box_to_wkt(w, s, e, n):
    return f'POLYGON(({w} {s},{e} {s},{e} {n},{w} {n},{w} {s}))'

def coverage_pct(geometry, b):
    try:
        foot = shape(geometry)
        if foot.is_empty: return None
        t = shp_box(*b)
        return round(100 * foot.intersection(t).area / t.area)
    except Exception:
        return None

def search_all(prod, wkt, chunks=4):
    edges = pd.date_range(pd.Timestamp(START), pd.Timestamp(END), periods=chunks + 1)
    got = {}
    for a, b in zip(edges[:-1], edges[1:]):
        res = asf.search(dataset='NISAR', processingLevel=prod, intersectsWith=wkt,
                         start=a.strftime('%Y-%m-%dT%H:%M:%S'), end=b.strftime('%Y-%m-%dT%H:%M:%S'),
                         dataMaturity='PROVISIONAL', maxResults=2000)
        for r in res:
            got[r.properties['sceneName']] = r
    return list(got.values())

results, rows = {}, []
for site, b in SITES.items():
    for prod in PRODUCTS:
        res = search_all(prod, box_to_wkt(*b))
        results[(site, prod)] = res
        for r in res:
            p = r.properties
            rows.append(dict(
                site=site, product=prod,
                start=str(p.get('startTime', ''))[:19], stop=str(p.get('stopTime', ''))[:19],
                track=p.get('pathNumber'), direction=p.get('flightDirection'),
                scene=p.get('sceneName'), box_covered_pct=coverage_pct(r.geometry, b)))

table = pd.DataFrame(rows)
table['track'] = pd.to_numeric(table['track'], errors='coerce')
by_scene = {r.properties.get('sceneName'): r for res in results.values() for r in res}

print('ALL scenes found, by track:')
print(table.groupby(['site', 'product', 'track', 'direction']).size().rename('scenes').to_string())

parts = [table[(table.site == s) & (table.track == pk['track']) & (table.direction == pk['direction'])]
         for s, pk in PICK.items()]
cand = pd.concat(parts)
cand = cand[cand.box_covered_pct.isna() | (cand.box_covered_pct >= MIN_COVER)]

print('\nKEPT (chosen track, box coverage >= 95%):')
print(cand.groupby(['site', 'product']).size().rename('scenes').to_string())
print('\nFirst and last date kept:')
print(cand.groupby(['site', 'product']).agg(first=('start', 'min'), last=('start', 'max')).to_string())

ERROR:asf_search:The asf-search module ecountered an error with CMR,and the following message was automatically reported to ASF:

"
message
"If you have any questions email uso@asf.alaska.edu
ERROR:asf_search:The asf-search module ecountered an error with CMR,and the following message was automatically reported to ASF:

"
message
"If you have any questions email uso@asf.alaska.edu
ERROR:asf_search:Results may be incomplete due to a search error. See ASF_LOGGER logging for more details.
ERROR:asf_search:The asf-search module ecountered an error with CMR,and the following message was automatically reported to ASF:

"
message
"If you have any questions email uso@asf.alaska.edu
ERROR:asf_search:The asf-search module ecountered an error with CMR,and the following message was automatically reported to ASF:

"
message
"If you have any questions email uso@asf.alaska.edu
ERROR:asf_search:Results may be incomplete due to a search error. See ASF_LOGGER logging for more details.
ERROR:asf_search:T

ALL scenes found, by track:
site           product  track  direction 
langtang       GCOV     48     DESCENDING    17
                        98     ASCENDING     23
                        149    DESCENDING     8
                        170    ASCENDING      9
               GOFF     48     DESCENDING    15
                        98     ASCENDING     21
                        149    DESCENDING     8
                        170    ASCENDING      9
               GUNW     48     DESCENDING    15
                        98     ASCENDING     23
                        149    DESCENDING     7
                        170    ASCENDING      9
suicide_basin  GCOV     34     ASCENDING     14
                        43     DESCENDING     8
                        63     ASCENDING      8
                        72     DESCENDING    12
                        135    ASCENDING      8
                        144    DESCENDING     8
               GOFF     34     ASCENDING     14
                  

# Stage 2

In [38]:
EVENT = {'langtang': '2026-08-26', 'suicide_basin': '2026-08-13'}

def label(r):
    ev = pd.Timestamp(EVENT[r.site])
    s = pd.to_datetime(str(r.start)[:19]); e = pd.to_datetime(str(r.stop)[:19])
    if e < ev: return 'before'
    if s >= ev: return 'after'
    return 'spans'

cand = cand.copy()
cand['when'] = cand.apply(label, axis=1)
cand['day'] = cand.start.str[:10]
cand['end_day'] = cand.stop.str[:10]
cand['pol'] = cand.scene.map(lambda s: ','.join(by_scene[s].properties.get('mainBandPolarization') or []))
cand['frame_cov'] = cand.scene.map(lambda s: by_scene[s].properties.get('frameCoverage'))
cand['pge'] = cand.scene.map(lambda s: by_scene[s].properties.get('pgeVersion'))

cand = (cand.sort_values('box_covered_pct', ascending=False, na_position='last')
            .drop_duplicates(['site', 'product', 'day', 'end_day'])
            .sort_values(['site', 'product', 'start']))

print('Scenes per site / product, before - spans - after the event:')
print(cand.groupby(['site', 'product', 'when']).size().unstack(fill_value=0)
          .reindex(columns=['before', 'spans', 'after'], fill_value=0))

print('\nAll kept scenes:')
print(cand[['site', 'product', 'when', 'start', 'stop', 'box_covered_pct', 'pol', 'frame_cov', 'pge']].to_string())

print('\nDays between consecutive GCOV scenes (expect 12, or 24+ across the Jul 27 - Aug 10 gap):')
for site in SITES:
    d = pd.to_datetime(cand[(cand.site == site) & (cand['product'] == 'GCOV')].day.unique())
    print(site, list(pd.Series(d).diff().dt.days.dropna().astype(int)))

Scenes per site / product, before - spans - after the event:
when                   before  spans  after
site          product                      
langtang      GCOV          5      0      3
              GOFF          5      1      3
              GUNW          5      1      3
suicide_basin GCOV          4      0      4
              GOFF          3      1      4
              GUNW          3      1      4

All kept scenes:
              site product    when                start                 stop  box_covered_pct    pol frame_cov       pge
13        langtang    GCOV  before  2026-06-20T23:39:21  2026-06-20T23:39:56              100  HH,HV      Full  R05.02.4
5         langtang    GCOV  before  2026-07-02T23:39:20  2026-07-02T23:39:55              100  HH,HV      Full  R05.02.4
23        langtang    GCOV  before  2026-07-14T23:39:20  2026-07-14T23:39:54              100  HH,HV      Full  R05.02.4
16        langtang    GCOV  before  2026-07-26T23:39:19  2026-07-26T23:39:54         

# Stage 3

In [39]:
def main_gb(p):
    b = p.get('bytes')
    if isinstance(b, dict) and b:
        return round(max((v.get('bytes') or 0) for v in b.values()) / 1e9, 2)
    return None

K = 3
picked = []
for (site, prod), g in cand.groupby(['site', 'product']):
    g = g.sort_values('start')
    if prod == 'GCOV':
        b, a = g[g.when == 'before'], g[g.when == 'after']
        k = min(K, len(b), len(a))
        if k < K: print(f'NOTE {site} GCOV: only {k} per side available')
        picked += [b.tail(k), a.head(k)]
    else:
        g12 = g[(pd.to_datetime(g.stop.str[:10]) - pd.to_datetime(g.day)).dt.days == 12]
        picked += [g12[g12.when == 'before'].tail(1),
                   g12[g12.when == 'spans'].head(1),
                   g12[g12.when == 'after'].head(1)]
sel = pd.concat(picked).copy()
sel['gb'] = sel.scene.map(lambda s: main_gb(by_scene[s].properties))
keep_scenes = set(sel.scene)

print(sel[['site', 'product', 'when', 'start', 'stop', 'gb']].to_string())
print('\nGB per site:'); print(sel.groupby('site').gb.sum().to_string())
print('Total GB:', round(sel.gb.sum(), 1))

              site product    when                start                 stop    gb
23        langtang    GCOV  before  2026-07-14T23:39:20  2026-07-14T23:39:54  7.05
16        langtang    GCOV  before  2026-07-26T23:39:19  2026-07-26T23:39:54  7.24
33        langtang    GCOV  before  2026-08-19T23:39:18  2026-08-19T23:39:53  7.24
26        langtang    GCOV   after  2026-08-31T23:39:18  2026-08-31T23:39:52  7.05
48        langtang    GCOV   after  2026-09-12T23:39:18  2026-09-12T23:39:52  7.05
42        langtang    GCOV   after  2026-09-24T23:39:18  2026-09-24T23:39:52  7.05
131       langtang    GOFF  before  2026-07-14T23:39:20  2026-07-26T23:39:54  1.09
142       langtang    GOFF   spans  2026-08-19T23:39:18  2026-08-31T23:39:52  1.10
134       langtang    GOFF   after  2026-08-31T23:39:18  2026-09-12T23:39:52  1.10
78        langtang    GUNW  before  2026-07-14T23:39:20  2026-07-26T23:39:54  2.34
89        langtang    GUNW   spans  2026-08-19T23:39:18  2026-08-31T23:39:52  2.34
81  

# Stage 4

In [40]:
sel_full = sel.copy()
sel = sel_full[sel_full.site == 'suicide_basin'].sort_values('gb').groupby('product').head(1)

In [41]:
import os, glob, shutil

CONFIRM = True
RAW = f'{ROOT}/raw'
LOCAL = '/content/dl'

def expected_bytes(p):
    b = p.get('bytes')
    if isinstance(b, dict) and b:
        return max((v.get('bytes') or 0) for v in b.values())
    return None

plan = sel.sort_values('gb').copy()
plan['exp'] = plan.scene.map(lambda s: expected_bytes(by_scene[s].properties))
plan['dest'] = plan.apply(lambda r: f"{RAW}/{r.site}/{r['product']}/{r.scene}.h5", axis=1)

def is_done(r):
    return os.path.exists(r.dest) and bool(r.exp) and os.path.getsize(r.dest) >= 0.99 * r.exp

done_mask = plan.apply(is_done, axis=1)
todo = plan[~done_mask]
print(f'{int(done_mask.sum())} files already in Drive, {len(todo)} to download '
      f'(about {todo.gb.sum():.1f} GB)')
print(f'Free space on Colab disk: {shutil.disk_usage("/").free/1e9:.0f} GB')

if not CONFIRM:
    print('\nNothing downloaded. Set CONFIRM = True and run this cell again to start.')
else:
    failed = []
    for i, (_, r) in enumerate(todo.iterrows(), 1):
        try:
            shutil.rmtree(LOCAL, ignore_errors=True); os.makedirs(LOCAL)
            os.makedirs(os.path.dirname(r.dest), exist_ok=True)
            by_scene[r.scene].download(path=LOCAL, session=session)
            f = glob.glob(LOCAL + '/*.h5')[0]
            if r.exp and os.path.getsize(f) < 0.99 * r.exp:
                raise RuntimeError('download looks incomplete')
            shutil.copyfile(f, r.dest + '.part')
            os.replace(r.dest + '.part', r.dest)
            print(f'[{i}/{len(todo)}] saved {r.site}/{r["product"]}/{r.scene}.h5 ({r.gb} GB)')
        except Exception as e:
            failed.append(r.scene)
            print(f'[{i}/{len(todo)}] FAILED {r.scene}: {str(e)[:120]}')
        finally:
            shutil.rmtree(LOCAL, ignore_errors=True)
    print('\nFinished.', len(failed), 'failed.')
    if failed: print('Run this cell again to retry:', failed)

3 files already in Drive, 0 to download (about 0.0 GB)
Free space on Colab disk: 200 GB

Finished. 0 failed.


In [42]:
sel = sel_full

In [43]:
import os, glob, shutil

CONFIRM = True
RAW = f'{ROOT}/raw'
LOCAL = '/content/dl'

def expected_bytes(p):
    b = p.get('bytes')
    if isinstance(b, dict) and b:
        return max((v.get('bytes') or 0) for v in b.values())
    return None

plan = sel.sort_values('gb').copy()
plan['exp'] = plan.scene.map(lambda s: expected_bytes(by_scene[s].properties))
plan['dest'] = plan.apply(lambda r: f"{RAW}/{r.site}/{r['product']}/{r.scene}.h5", axis=1)

def is_done(r):
    return os.path.exists(r.dest) and bool(r.exp) and os.path.getsize(r.dest) >= 0.99 * r.exp

done_mask = plan.apply(is_done, axis=1)
todo = plan[~done_mask]
print(f'{int(done_mask.sum())} files already in Drive, {len(todo)} to download '
      f'(about {todo.gb.sum():.1f} GB)')
print(f'Free space on Colab disk: {shutil.disk_usage("/").free/1e9:.0f} GB')

if not CONFIRM:
    print('\nNothing downloaded. Set CONFIRM = True and run this cell again to start.')
else:
    failed = []
    for i, (_, r) in enumerate(todo.iterrows(), 1):
        try:
            shutil.rmtree(LOCAL, ignore_errors=True); os.makedirs(LOCAL)
            os.makedirs(os.path.dirname(r.dest), exist_ok=True)
            by_scene[r.scene].download(path=LOCAL, session=session)
            f = glob.glob(LOCAL + '/*.h5')[0]
            if r.exp and os.path.getsize(f) < 0.99 * r.exp:
                raise RuntimeError('download looks incomplete')
            shutil.copyfile(f, r.dest + '.part')
            os.replace(r.dest + '.part', r.dest)
            print(f'[{i}/{len(todo)}] saved {r.site}/{r["product"]}/{r.scene}.h5 ({r.gb} GB)')
        except Exception as e:
            failed.append(r.scene)
            print(f'[{i}/{len(todo)}] FAILED {r.scene}: {str(e)[:120]}')
        finally:
            shutil.rmtree(LOCAL, ignore_errors=True)
    print('\nFinished.', len(failed), 'failed.')
    if failed: print('Run this cell again to retry:', failed)


24 files already in Drive, 0 to download (about 0.0 GB)
Free space on Colab disk: 200 GB

Finished. 0 failed.


# Stage 5

In [ ]:
import h5py, glob, os
import pandas as pd
pd.set_option('display.width', 250); pd.set_option('display.max_rows', 100)

SPEC = {
    'GCOV': dict(grid='science/LSAR/GCOV/grids/frequencyA', main='HHHH'),
    'GOFF': dict(grid='science/LSAR/GOFF/grids/frequencyA/pixelOffsets', main='HH/layer1/slantRangeOffset'),
    'GUNW': dict(grid='science/LSAR/GUNW/grids/frequencyA/unwrappedInterferogram', main='HH/unwrappedPhase'),
}
def txt(v):
    v = v[()]
    return v.decode() if isinstance(v, bytes) else str(v)

inv = []
for _, r in plan.iterrows():
    sp = SPEC[r['product']]
    with h5py.File(r.dest, 'r') as h:
        g = h[sp['grid']]
        x, y = g['xCoordinates'], g['yCoordinates']
        sw = f"science/LSAR/{r['product']}/metadata/processingInformation/algorithms/softwareVersion"
        inv.append(dict(site=r.site, product=r['product'], when=r.when, start=r.start[:10], stop=r.stop[:10],
                        epsg=int(g['projection'][()]), dx=float(g['xCoordinateSpacing'][()]), dy=float(g['yCoordinateSpacing'][()]),
                        shape=g[sp['main']].shape, x0=float(x[0]), x1=float(x[-1]), y0=float(y[0]), y1=float(y[-1]),
                        version=txt(h[sw]) if sw in h else None))
inv = pd.DataFrame(inv)
print(inv.drop(columns=['when']).to_string(index=False))

print('\nDo all scenes of a site/product share the same grid?')
cols = ['epsg', 'dx', 'dy', 'shape', 'x0', 'y0']
print(inv.astype({'shape': str}).groupby(['site', 'product'])[cols].nunique().to_string())

# Stage 6

In [ ]:
import numpy as np, h5py, os
from pyproj import Transformer
import matplotlib.pyplot as plt

def box_in_utm(box, epsg):
    t = Transformer.from_crs(4326, epsg, always_xy=True)
    w, s, e, n = box
    xs, ys = zip(*[t.transform(x, y) for x, y in [(w, s), (e, s), (e, n), (w, n)]])
    return min(xs), max(xs), min(ys), max(ys)

def window(x, y, xmin, xmax, ymin, ymax):
    ix = np.where((x >= xmin) & (x <= xmax))[0]
    iy = np.where((y >= ymin) & (y <= ymax))[0]
    if len(ix) == 0 or len(iy) == 0:
        raise RuntimeError('box is outside this grid')
    return iy.min(), iy.max() + 1, ix.min(), ix.max() + 1

L = 'science/LSAR/'
OFF = [f'HH/layer{k}/{n}' for k in (1, 2, 3) for n in ('slantRangeOffset', 'alongTrackOffset', 'snr', 'correlationSurfacePeak')]
BLOCKS = {
    'GCOV': [('g', L + 'GCOV/grids/frequencyA', ['HHHH', 'HVHV', 'mask'])],
    'GOFF': [('o', L + 'GOFF/grids/frequencyA/pixelOffsets', OFF + ['mask'])],
    'GUNW': [('u', L + 'GUNW/grids/frequencyA/unwrappedInterferogram',
              ['HH/unwrappedPhase', 'HH/coherenceMagnitude', 'HH/connectedComponents', 'mask']),
             ('w', L + 'GUNW/grids/frequencyA/wrappedInterferogram', ['HH/coherenceMagnitude', 'mask'])],
}

def crop_file(path, product, box):
    out = {}
    with h5py.File(path, 'r') as h:
        for prefix, grp, names in BLOCKS[product]:
            g = h[grp]
            epsg = int(g['projection'][()])
            x, y = g['xCoordinates'][:], g['yCoordinates'][:]
            j0, j1, i0, i1 = window(x, y, *box_in_utm(box, epsg))
            out[prefix + '__x'], out[prefix + '__y'] = x[i0:i1], y[j0:j1]
            out[prefix + '__epsg'] = np.array(epsg)
            for n in names:
                out[prefix + '__' + n.replace('/', '_')] = g[n][j0:j1, i0:i1]
    return out

def crop_signal(site, product):
    rows = plan[(plan.site == site) & (plan['product'] == product)].sort_values('start')
    print(f'--- {site} {product}: {len(rows)} files')
    for _, r in rows.iterrows():
        dest = f"{ROOT}/crops/{site}/{product}/{r.scene}.npz"
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        if os.path.exists(dest):
            print('  already done', r.start[:10]); continue
        try:
            arrs = crop_file(r.dest, product, SITES[site])
            tmp = dest[:-4] + '.part.npz'
            np.savez_compressed(tmp, **arrs)
            os.replace(tmp, dest)
            shape = next(v.shape for v in arrs.values() if v.ndim == 2)
            print(f'  saved {r.start[:10]} ({r["when"]})  crop {shape}  {os.path.getsize(dest)/1e6:.0f} MB')
        except Exception as e:
            print(f'  FAILED {r.start[:10]}: {str(e)[:150]}')

def quicklook(site):
    g = plan[(plan.site == site) & (plan['product'] == 'GCOV')].sort_values('start')
    pair = [g[g.when == 'before'].iloc[-1], g[g.when == 'after'].iloc[0]]
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    for a, r in zip(ax, pair):
        z = np.load(f"{ROOT}/crops/{site}/GCOV/{r.scene}.npz")
        hh = z['g__HHHH']; st = max(1, max(hh.shape) // 1200); hh = hh[::st, ::st]
        db = 10 * np.log10(np.where(hh > 0, hh, np.nan))
        lo, hi = np.nanpercentile(db, [2, 98])
        a.imshow(db, cmap='gray', vmin=lo, vmax=hi); a.set_title(f'{site} {r["when"]} {r.start[:10]}'); a.axis('off')
    plt.tight_layout(); plt.savefig(f'{ROOT}/crops/{site}_quicklook.png', dpi=100); plt.show()

# Suicide Basin

In [ ]:
crop_signal('suicide_basin', 'GCOV'); quicklook('suicide_basin')

In [ ]:
def blockmean(a, k):
    h, w = (a.shape[0] // k) * k, (a.shape[1] // k) * k
    return a[:h, :w].reshape(h // k, k, w // k, k).mean(axis=(1, 3))

def to_db(a):
    return 10 * np.log10(np.where(a > 0, a, np.nan))

def stretch(a):
    lo, hi = np.nanpercentile(a, [2, 98])
    return np.clip((a - lo) / (hi - lo), 0, 1)

def colorlook(site):
    g = plan[(plan.site == site) & (plan['product'] == 'GCOV')].sort_values('start')
    rb, ra = g[g.when == 'before'].iloc[-1], g[g.when == 'after'].iloc[0]
    zb = np.load(f"{ROOT}/crops/{site}/GCOV/{rb.scene}.npz")
    za = np.load(f"{ROOT}/crops/{site}/GCOV/{ra.scene}.npz")
    k = max(1, max(zb['g__HHHH'].shape) // 1000)
    hhb, hha = to_db(blockmean(zb['g__HHHH'], k)), to_db(blockmean(za['g__HHHH'], k))
    hvb, hva = to_db(blockmean(zb['g__HVHV'], k)), to_db(blockmean(za['g__HVHV'], k))

    lo, hi = np.nanpercentile(hhb, [2, 98])
    fig, ax = plt.subplots(1, 2, figsize=(13, 6))
    for a_, img, t in [(ax[0], hhb, f'before {rb.start[:10]}'), (ax[1], hha, f'after {ra.start[:10]}')]:
        im = a_.imshow(img, cmap='viridis', vmin=lo, vmax=hi); a_.set_title(f'{site} HH, {t}'); a_.axis('off')
    fig.colorbar(im, ax=ax, shrink=0.8, label='brightness (dB), same scale in both')
    plt.show()

    fig, ax = plt.subplots(1, 2, figsize=(13, 6))
    for a_, d, t in [(ax[0], hha - hhb, 'HH'), (ax[1], hva - hvb, 'HV')]:
        lim = min(10, np.nanpercentile(np.abs(d), 98))
        im = a_.imshow(d, cmap='bwr', vmin=-lim, vmax=lim); a_.set_title(f'{site} {t} change (after minus before)'); a_.axis('off')
        fig.colorbar(im, ax=a_, shrink=0.8, label='dB   blue = darker after, red = brighter after')
    plt.show()

    rgb = np.dstack([stretch(hha), stretch(hva), stretch(hha - hva)])
    plt.figure(figsize=(7, 7)); plt.imshow(np.nan_to_num(rgb)); plt.axis('off')
    plt.title(f'{site} after: red = HH, green = HV, blue = HH minus HV'); plt.show()

colorlook('suicide_basin')

In [ ]:
crop_signal('suicide_basin', 'GOFF')

In [ ]:
crop_signal('suicide_basin', 'GUNW')

# Langtang

In [ ]:
crop_signal('langtang', 'GCOV'); quicklook('langtang')

In [ ]:
crop_signal('langtang', 'GOFF')

In [ ]:
crop_signal('langtang', 'GUNW')